# 01 — Zero-Shot Baseline (Phase 2, ~1 credit)

Measures what COCO-pretrained RF-DETR already does **before** any training, so
Phase 3's gain is measurable rather than assumed.

**Expected result:** people detected reliably (`person` is COCO class 1);
forklifts **missed or mislabeled** as `truck`/`car` — COCO has no forklift class.
That gap is precisely what fine-tuning closes.

**Acceptance check:** ≥80% of visible people have boxes; forklifts wrong or
absent (expected). If *people* are badly missed, the footage is unusual (extreme
angle, very dark) — collect more representative video before training, because
fine-tuning will not rescue unrepresentative data.

In [ ]:
# Colab setup. T4 GPU ONLY — never A100 (2 credits/hr vs 12).
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Pins match requirements.txt; see src/detector.py for the API drift each guards.
!pip install -q "rfdetr>=1.9.0,<2.0.0" "supervision>=0.29,<0.30" trackers rtmlib onnxruntime-gpu

from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/warehouse-safety'

import sys
sys.path.insert(0, PROJECT)          # so `from src...` resolves to the mirrored repo
print('project:', PROJECT)

In [ ]:
import glob, os, cv2, supervision as sv
from src.detector import RFDetrDetector, coco_class_names

COCO_CLASSES = coco_class_names()     # rfdetr moved this module in 1.9.0
detector = RFDetrDetector(weights=None, threshold=0.5)   # None = COCO weights

OUT = f'{PROJECT}/outputs/baseline'
os.makedirs(OUT, exist_ok=True)

frames = sorted(glob.glob(f'{PROJECT}/data/frames/*.jpg'))
print(f'{len(frames)} frames available; annotating the first 30')
assert frames, 'no frames — run src/extract_frames.py first (Phase 1)'

In [ ]:
box_ann, lab_ann = sv.BoxAnnotator(), sv.LabelAnnotator()
person_hits = 0

for path in frames[:30]:
    bgr = cv2.imread(path)
    # RFDetrDetector converts BGR->RGB internally. Calling model.predict()
    # directly on an OpenCV frame is the #1 silent accuracy bug in this project.
    dets = detector(bgr)
    labels = [f'{COCO_CLASSES.get(int(c), c)} {conf:.2f}'
              for c, conf in zip(dets.class_id, dets.confidence)]
    person_hits += sum(1 for c in dets.class_id if int(c) == 1)
    out = lab_ann.annotate(box_ann.annotate(bgr.copy(), dets), dets, labels)
    cv2.imwrite(f'{OUT}/baseline_{os.path.basename(path)}', out)
    print(os.path.basename(path), labels)

print(f'\ntotal person detections across 30 frames: {person_hits}')
print(f'annotated images -> {OUT}')
print('\nNow OPEN those images and count by eye. This check is visual on purpose:')
print('there are no labels yet, so no metric can be computed — that is what Phase 3 is for.')

## Record before moving on

Write down the person-detection rate you observed by eye. Phase 3's acceptance
check compares against it: **if person accuracy drops after fine-tuning, that is
catastrophic forgetting** — retrain with `lr=5e-5`. Without this baseline number
you cannot tell that it happened.

Then **Runtime → Disconnect**. Idle sessions burn credits.